Nhật ký sau đây được GitHub Copilot Chat tự động tạo và chỉ dành cho việc thiết lập ban đầu


# Giới thiệu về Kỹ thuật Đề bài
Kỹ thuật đề bài là quá trình thiết kế và tối ưu hóa các đề bài cho các nhiệm vụ xử lý ngôn ngữ tự nhiên. Nó bao gồm việc lựa chọn đề bài phù hợp, điều chỉnh các tham số của chúng và đánh giá hiệu suất của chúng. Kỹ thuật đề bài rất quan trọng để đạt được độ chính xác và hiệu quả cao trong các mô hình NLP. Trong phần này, chúng ta sẽ khám phá các kiến thức cơ bản về kỹ thuật đề bài sử dụng các mô hình OpenAI để thăm dò.


### Bài tập 1: Phân tách từ (Tokenization)
Khám phá Phân tách từ sử dụng tiktoken, một bộ phân tách từ nhanh mã nguồn mở từ OpenAI
Xem [OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb?WT.mc_id=academic-105485-koreyst) để có thêm ví dụ.


In [ ]:
# EXERCISE:
# 1. Run the exercise as is first
# 2. Change the text to any prompt input you want to use & re-run to see tokens

import tiktoken

# Define the prompt you want tokenized
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

# Set the model you want encoding for
encoding = tiktoken.encoding_for_model("gpt-4o")

# Encode the text - gives you the tokens in integer form
tokens = encoding.encode(text)
print(tokens);

# Decode the integers to see what the text versions look like
[encoding.decode_single_token_bytes(token) for token in tokens]


### Bài tập 2: Xác thực Cấu hình Khóa API OpenAI

Chạy mã dưới đây để kiểm tra xem điểm cuối OpenAI của bạn đã được thiết lập chính xác chưa. Mã chỉ thử một lệnh nhắc đơn giản và xác thực kết quả hoàn thành. Đầu vào `oh say can you see` nên được hoàn thành theo hướng `by the dawn's early light..`


In [ ]:
# Uses the OpenAI client with the Responses API.
# See https://platform.openai.com/docs/api-reference/responses

import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI()

deployment="gpt-4o-mini"

def get_completion(prompt):
    response = client.responses.create(
        model=deployment,
        input=prompt,
        temperature=0, # this is the degree of randomness of the model's output
        max_output_tokens=1024,
        store=False,
    )
    return response.output_text

## ---------- Call the helper method

### 1. Set primary content or prompt text
text = f"""
oh say can you see
"""

### 2. Use that in the prompt template below
prompt = f"""
```{text}```
"""

## 3. Run the prompt
response = get_completion(prompt)
print(response)


### Bài tập 3: Sự bịa đặt
Khám phá điều gì xảy ra khi bạn yêu cầu LLM trả về các hoàn thành cho một đề bài về một chủ đề có thể không tồn tại, hoặc về các chủ đề mà nó có thể không biết vì chúng nằm ngoài tập dữ liệu đã được huấn luyện trước (gần đây hơn). Xem cách phản hồi thay đổi nếu bạn thử một đề bài khác, hoặc một mô hình khác.


In [ ]:

## Set the text for simple prompt or primary content
## Prompt shows a template format with text in it - add cues, commands etc if needed
## Run the completion 
text = f"""
generate a lesson plan on the Martian War of 2076.
"""

prompt = f"""
```{text}```
"""

response = get_completion(prompt)
print(response)

### Bài tập 4: Dựa trên Hướng dẫn 
Sử dụng biến "text" để đặt nội dung chính 
và biến "prompt" để cung cấp một hướng dẫn liên quan đến nội dung chính đó.

Ở đây chúng ta yêu cầu mô hình tóm tắt văn bản cho một học sinh lớp hai


In [ ]:
# Test Example
# https://platform.openai.com/playground/p/default-summarize

## Example text
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

## Set the prompt
prompt = f"""
Summarize content you are provided with for a second-grade student.
```{text}```
"""

## Run the prompt
response = get_completion(prompt)
print(response)

### Bài tập 5: Lời nhắc phức tạp 
Thử một yêu cầu có các tin nhắn hệ thống, người dùng và trợ lý 
Hệ thống đặt ngữ cảnh cho trợ lý
Tin nhắn của Người dùng & Trợ lý cung cấp ngữ cảnh cuộc trò chuyện đa lượt

Lưu ý cách cá tính trợ lý được đặt là "mỉa mai" trong ngữ cảnh hệ thống. 
Thử sử dụng một ngữ cảnh cá tính khác. Hoặc thử một chuỗi tin nhắn đầu vào/đầu ra khác


In [ ]:
response = client.responses.create(
    model=deployment,
    input=[
        {"role": "system", "content": "You are a sarcastic assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},
        {"role": "assistant", "content": "Who do you think won? The Los Angeles Dodgers of course."},
        {"role": "user", "content": "Where was it played?"}
    ],
    store=False,
)
print(response.output_text)


### Bài tập: Khám phá Trực giác của bạn
Các ví dụ trên cung cấp cho bạn những mẫu mà bạn có thể sử dụng để tạo các lời nhắc mới (đơn giản, phức tạp, hướng dẫn, v.v.) - hãy thử tạo các bài tập khác để khám phá một số ý tưởng khác mà chúng ta đã bàn như ví dụ, gợi ý và nhiều hơn nữa.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
